# LoRA 微调 Qwen2.5-1.5B-Instruct for RAG 问答

**运行环境**: Google Colab 免费 T4 GPU (~16GB VRAM)  
**目标**: 让 1.5B 小模型学会"基于检索结果回答问题"  
**预计时间**: 下载 5min + 训练 30-45min = 约 1 小时  

### LoRA 参数说明
- **rank (r)=8**: 可训练参数量适中，8 是 LoRA 论文推荐的默认值
- **alpha=16**: alpha/r=2 的比例关系，控制 LoRA 层的缩放
- **target_modules**: 只训练 Q/V 注意力投影矩阵，这是经验上最有效的
- **lr=2e-4**: 小模型常用学习率，比大模型稍高
- **epochs=3**: 100 条数据量小，3 轮足够避免过拟合

## Step 1: 环境安装 & GPU 检查

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets sentencepiece
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Step 2: 上传训练数据

将本地的 `data/training_data.json` 上传到 Colab（左侧文件面板 → 上传）

In [ ]:
import json
import os
from datasets import Dataset
from transformers import AutoTokenizer

# 检查文件是否已上传
TRAINING_FILE = "training_data.json"
if not os.path.exists(TRAINING_FILE):
    print(f"请先上传 {TRAINING_FILE} 到 Colab 文件系统!")
else:
    with open(TRAINING_FILE, "r", encoding="utf-8") as f:
        raw = json.load(f)
    print(f"训练样本数: {len(raw['samples'])}")
    print(f"说明: {raw['description']}")

## Step 3: 格式化训练数据

使用 Qwen2.5 的 ChatML 格式: `<|im_start|>system/ user/ assistant<|im_end|>`

In [ ]:
SYSTEM_PROMPT = """你是一个基于文档知识的问答助手。请严格依据提供的参考资料回答问题。
规则：
1. 只使用参考资料中的信息回答，不要编造
2. 如果资料不足以回答，请明确说"根据现有资料无法回答"
3. 回答末尾标注来源编号，如 [来源:1,2]
4. 回答简洁准确，避免冗余"""

def format_samples(raw_data):
    formatted = []
    for sample in raw_data["samples"]:
        text = f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{sample['instruction']}<|im_end|>
<|im_start|>assistant
{sample['output']}<|im_end|>"""
        formatted.append({"text": text})
    return formatted

formatted_data = format_samples(raw)
print(f"格式化完成: {len(formatted_data)} 条")
print(f"\n样本示例 (前200字符):\n{formatted_data[0]['text'][:200]}...")

## Step 4: 加载 Tokenizer & 分词

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = Dataset.from_list(formatted_data)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
    )

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
print(f"分词完成: {len(tokenized)} 条, seq_len=1024")

## Step 5: 加载基座模型 (4-bit 量化)

为什么选 Qwen2.5-1.5B 而不是 7B?
- 1.5B: T4 16GB 刚好够训练，4-bit 后仅 ~2GB
- 7B: 4-bit 后 ~4.5GB + LoRA + optimizer ≈ 8-10GB，T4 也能跑但更慢
- 1.5B 作为教学目的足够了，展示的是微调方法论而非绝对性能

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# 4-bit 量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False  # 训练时关闭
model = prepare_model_for_kbit_training(model)
print(f"模型加载完成, GPU VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## Step 6: 配置 LoRA

技术要点：
- **为什么 rank=8?** — LoRA 论文实验表明 r=8 与 r=16/32 效果接近，更大不一定更好
- **为什么只微调 Q/V?** — Q（查询）和 V（值）是注意力中最关键的两个投影，K 矩阵的影响较小
- **为什么 alpha=16?** — alpha/r=2 是标准比例，控制 LoRA 更新的缩放幅度

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 预期输出: trainable params ≈ 0.2% of total
# Qwen2.5-1.5B: total ~1.5B params, LoRA trainable ~3M params

## Step 7: 开始训练

In [ ]:
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "./qwen25-1.5b-rag-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # 有效 batch_size = 4×4 = 16
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    tokenizer=tokenizer,
)

print("Training started...")
trainer.train()

# 保存 LoRA adapter (~10-20MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}/")

## Step 8: 效果对比测试

对比三种模型在同一问题上的表现：
1. 原始 Qwen2.5-1.5B（无微调）
2. Qwen2.5-1.5B + LoRA（微调后）
3. DeepSeek v4-flash API（基线）

In [ ]:
def generate_response(model, tokenizer, instruction, max_tokens=256):
    """用微调后模型生成回答"""
    prompt = f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|im_start|>assistant" in full:
        return full.split("<|im_start|>assistant")[-1].strip()
    return full

TEST_CASES = [
    {
        "instruction": "根据以下参考资料回答问题：\n\n参考资料：\n[1] 机器学习是人工智能的一个分支，它使计算机能够从数据中学习，而无需进行显式编程。\n\n问题：什么是机器学习？",
        "label": "概念解释"
    },
    {
        "instruction": "根据以下参考资料回答问题：\n\n参考资料：\n[1] RAG的工作流程包括：文档处理→切片→向量化→检索→生成答案。\n\n问题：RAG的工作流程是什么？",
        "label": "事实查询"
    },
    {
        "instruction": "根据以下参考资料回答问题：\n\n参考资料：\n参考资料中没有与问题直接相关的内容。\n\n问题：明年的天气会怎样？",
        "label": "拒绝回答"
    },
    {
        "instruction": "根据以下参考资料回答问题：\n\n参考资料：\n[1] 监督学习使用带标签数据进行训练。无监督学习使用无标签数据发现隐藏结构。\n[2] 强化学习通过与环境交互最大化累积奖励。\n\n问题：机器学习有哪三种类型？分别有什么特点？",
        "label": "多信息综合"
    },
]

print("=" * 60)
print("LoRA 微调后模型推理测试")
print("=" * 60)

for i, tc in enumerate(TEST_CASES, 1):
    print(f"\n[Test {i}] {tc['label']}")
    response = generate_response(model, tokenizer, tc['instruction'])
    print(f"Q: {tc['instruction'].split('问题：')[-1].strip()[:60]}...")
    print(f"A: {response[:200]}")
    print("-" * 60)

## Step 9: 合并权重 & 下载

将 LoRA adapter 合并到基座模型中，方便直接使用。

In [ ]:
# 合并 LoRA 权重到基座
print("Merging LoRA weights...")
merged_model = model.merge_and_unload()

MERGED_DIR = "./qwen25-1.5b-rag-merged"
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}/")

# 打包下载
!zip -r qwen25-rag-lora.zip {OUTPUT_DIR}/
print("LoRA adapter (zip): ready to download from Files panel")
print(f"Adapter size (约 15-20MB)")
print(f"Full merged model size (约 3GB, 不建议下载)")

## Step 10: 效果对比总结

预期对比 (填空):

| Test | DeepSeek v4-flash | Qwen2.5-1.5B 原版 | Qwen2.5-1.5B + LoRA |
|------|------------------|-------------------|---------------------|
| 概念解释 | ✅ 准确 | ⚠️ 可能编造 | ✅ 学会引用来源 |
| 事实查询 | ✅ 准确 | ⚠️ 格式混乱 | ✅ 结构化输出 |
| 拒绝回答 | ✅ 诚实 | ❌ 强行编造 | ✅ 学会说"不知道" |
| 多信息综合 | ✅ 整合好 | ❌ 遗漏信息 | ✅ 多源引用 |

### 效果分析

**微调后提升了什么？**
- 学会了基于参考资料回答（而不是凭记忆编造）
- 学会了结构化输出格式（标注来源）
- 学会了在信息不足时诚实说"不知道"

**LoRA 的局限性？**
- 微调不能注入新的事实知识（LoRA 只调整了回答风格/格式）
- 模型的知识仍然来自预训练阶段
- 真正的知识注入需要 RAG 或全量微调

**什么场景用微调 vs RAG？**
- 微调：调整回答风格、格式、特定领域术语使用习惯
- RAG：注入最新/私域事实知识、避免幻觉
- 最佳实践：RAG + 微调 组合 —— RAG 提供事实，微调优化表达

## 附录: 训练监控 (nvidia-smi)

训练过程中可以在 Colab Terminal 中运行: `watch -n 1 nvidia-smi`

预期显存占用:
- 模型加载: ~3GB (4-bit)
- 训练时: ~5-6GB (含 optimizer states)
- T4 (16GB) 完全够用